# Phase 2: build the dataset from Google Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhavilkawdia5-ux/leaf_disease_detection/blob/main/ml/notebooks/phase2_dataset.ipynb)

This runs `ml/src/dataset.py` (Phase 2 in `docs/SYSTEM_DESIGN.md`) on the raw datasets
in your Google Drive. It:

1. merges all public datasets into the KrishiCare classes from `ml/classes.yaml`,
2. splits them train / val / test 70 / 15 / 15 (the test split is never trained on),
3. prints image counts per class and per split,
4. makes one sheet of 25 sample photos per class, so you can check the labels by eye,
5. saves `processed.zip` and `phase2_counts.txt` to Drive for Phase 3.

**Before you start:** put each dataset in `MyDrive/leafsight/data/raw/<folder>/`
(folder names are in `ml/sources.yaml` and `docs/datasets.md`). Zips can stay zipped.
No GPU is needed. Run each cell with **Shift + Enter**, top to bottom.

## 1. Connect Google Drive and get the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
if os.path.isdir("/content/leaf_disease_detection"):
    !git -C /content/leaf_disease_detection pull -q
else:
    !git clone -q https://github.com/bhavilkawdia5-ux/leaf_disease_detection.git /content/leaf_disease_detection
%cd /content/leaf_disease_detection
!git log --oneline -1

## 2. Install packages

In [ ]:
!pip install -q pyyaml pyarrow pillow

## 3. Check the raw datasets in Drive

Every folder name should be listed in `ml/sources.yaml`. A missing one is fine; it is
reported as "Not downloaded yet".

In [ ]:
from pathlib import Path

DRIVE_DATA = Path("/content/drive/MyDrive/leafsight/data")
raw = DRIVE_DATA / "raw"
assert raw.is_dir(), f"Not found: {raw}. Put the datasets there first."
for p in sorted(raw.iterdir()):
    print(p.name)

## 4. Merge and count

The script reads raw data from Drive but writes the processed images to this Colab
machine first (much faster than writing thousands of small files to Drive).
It takes about 5 to 15 minutes. Read the **Notes** at the end of the output.

In [ ]:
WORK = Path("/content/data")
WORK.mkdir(exist_ok=True)
for name in ("raw", "field_test"):          # read from Drive, never changed
    link, target = WORK / name, DRIVE_DATA / name
    if target.is_dir() and not link.exists():
        link.symlink_to(target)

!python ml/src/dataset.py --data /content/data --preview | tee /content/phase2_counts.txt

## 5. Save the results to Drive

Phase 3 trains from `processed.zip`. `phase2_counts.txt` is the count table above.

In [ ]:
import shutil

shutil.copy("/content/phase2_counts.txt", DRIVE_DATA / "phase2_counts.txt")
zip_path = shutil.make_archive("/content/processed", "zip", WORK / "processed")
shutil.copy(zip_path, DRIVE_DATA / "processed.zip")
print("Saved to", DRIVE_DATA, ":", round(os.path.getsize(zip_path) / 1e6), "MB zip")

## 6. Check the labels by eye

Each sheet shows 25 photos from one class. For each sheet, look for:

- a photo that clearly shows a different disease, or a healthy leaf in a disease class
- something that is not a leaf of that crop (fruit, a hand, another plant, a collage)
- the same photo many times

Write down the class name and what looks wrong. Until an expert is available, this
replaces the expert label check (`docs/datasets.md`, "Workflow without field visits").

In [ ]:
from IPython.display import Image as Show, display

for sheet in sorted((WORK / "processed" / "preview").glob("*.jpg")):
    print(sheet.stem)
    display(Show(filename=str(sheet), width=480))

## 7. What to do next

1. Copy all the text printed by step 4 (or open `MyDrive/leafsight/data/phase2_counts.txt`).
2. Paste it to Claude Code with your notes from step 6, and ask it to update
   `docs/datasets.md` with the new counts.
3. Then tick the Phase 2 "Done when" checks.